<a href="https://colab.research.google.com/github/Chiranthan7/ECG-and-EEG-based-Biometric-Encryption-using-Machine-Learning/blob/main/FINAL_CAP_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wfdb pywavelets pycryptodome

import os
import numpy as np
import pandas as pd
import wfdb
import pywt
from scipy.signal import welch
from scipy.stats import entropy as scipy_entropy

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad, unpad
import hashlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 102.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [ ]:

BASE_DIR = "/content/drive/MyDrive/ECG EEG DATASET"


PERSON_IDS_TRAIN = [1, 2, 3, 4, 5]


PERSON_FOLDERS = {
    pid: os.path.join(BASE_DIR, f"Person_{pid:02d}")
    for pid in PERSON_IDS_TRAIN
}

RECORDS_PER_PERSON = ["rec_1", "rec_2"]  # adjust if you have more/other


FS = 500
PRE_SAMPLES = 200
POST_SAMPLES = 200
BEAT_LEN = PRE_SAMPLES + POST_SAMPLES


In [ ]:

def extract_beats_from_record(record_path, fs=FS, pre=PRE_SAMPLES, post=POST_SAMPLES):

    rec = wfdb.rdrecord(record_path)
    ann = wfdb.rdann(record_path, "atr")

    sig = rec.p_signal[:, 0]  # use channel 0
    r_peaks = ann.sample

    beats = []
    rr_intervals = []
    last_r_time = None

    for r in r_peaks:
        start = max(0, r - pre)
        end = min(len(sig), r + post)

        beat = sig[start:end]

        # Pad to fixed length at borders
        if len(beat) < pre + post:
            pad_left = pre - (r - start)
            pad_right = (pre + post) - len(beat) - pad_left
            beat = np.pad(beat, (pad_left, pad_right), mode="constant")

        beats.append(beat)

        if last_r_time is not None:
            rr_intervals.append((r - last_r_time) / fs)
        else:
            rr_intervals.append(np.nan)

        last_r_time = r

    return beats, r_peaks, np.array(rr_intervals)

In [ ]:

# ---- Time-domain features ----
def time_domain_features(beat, fs=FS, pre=PRE_SAMPLES):
    x = beat.astype(float)
    N = len(x)
    dx = np.diff(x)

    mean = np.mean(x)
    std = np.std(x)
    var = np.var(x)
    median = np.median(x)
    x_max = np.max(x)
    x_min = np.min(x)
    ptp = x_max - x_min
    rms = np.sqrt(np.mean(x**2))
    energy = np.sum(x**2)
    zc = np.sum(x[:-1] * x[1:] < 0) / N

    dx_mean = np.mean(dx)
    dx_energy = np.sum(dx**2)

    if pre < len(x):
        r_amp = x[pre]
        k = 10
        pre_idx = max(0, pre - k)
        post_idx = min(len(x) - 1, pre + k)
        pre_slope = (x[pre] - x[pre_idx]) / (pre - pre_idx + 1e-6)
        post_slope = (x[post_idx] - x[pre]) / (post_idx - pre + 1e-6)
    else:
        r_amp = np.nan
        pre_slope = np.nan
        post_slope = np.nan

    return [
        mean, std, var, median, x_max, x_min, ptp,
        rms, energy, zc, dx_mean, dx_energy,
        r_amp, pre_slope, post_slope
    ]

# ---- Frequency-domain features ----
def frequency_domain_features(beat, fs=FS):
    x = beat.astype(float)
    f, Pxx = welch(x, fs=fs, nperseg=min(256, len(x)))

    total_power = np.sum(Pxx)
    if total_power <= 0:
        total_power = 1e-12

    dom_idx = np.argmax(Pxx)
    dom_freq = f[dom_idx]

    def band_power(fmin, fmax):
        idx = np.logical_and(f >= fmin, f <= fmax)
        return np.sum(Pxx[idx])

    p_0_5   = band_power(0, 5)
    p_5_15  = band_power(5, 15)
    p_15_40 = band_power(15, 40)

    centroid = np.sum(f * Pxx) / total_power
    mean_freq = centroid

    # Safe median frequency
    cum_power = np.cumsum(Pxx)
    half_power = total_power / 2
    idx_med = np.searchsorted(cum_power, half_power)
    if idx_med >= len(f):
        idx_med = len(f) - 1
    median_freq = f[idx_med]

    Pxx_norm = Pxx / total_power
    spec_entropy = scipy_entropy(Pxx_norm + 1e-12, base=2)

    gm = np.exp(np.mean(np.log(Pxx + 1e-12)))
    am = np.mean(Pxx)
    spec_flatness = gm / (am + 1e-12)

    return [
        total_power, dom_freq,
        p_0_5, p_5_15, p_15_40,
        centroid, mean_freq, median_freq,
        spec_entropy, spec_flatness
    ]

# ---- Wavelet-domain features ----
def wavelet_features(beat, wavelet="db4", level=4):
    x = beat.astype(float)
    coeffs = pywt.wavedec(x, wavelet, level=level)
    feats = []
    for c in coeffs:
        c = np.asarray(c)
        mean_c = np.mean(c)
        var_c = np.var(c)
        energy_c = np.sum(c**2)
        ps = (c**2) / (energy_c + 1e-12)
        ent_c = scipy_entropy(ps + 1e-12, base=2)
        feats.extend([mean_c, var_c, energy_c, ent_c])
    return feats  # 5 subbands × 4 = 20 features

# ---- Non-linear features ----

def _phi(x, m, r):
    N = len(x)
    if N <= m + 1:
        return 1e-12
    X = np.array([x[i:i+m] for i in range(N - m + 1)])
    C = np.sum(
        np.max(np.abs(X[:, None, :] - X[None, :, :]), axis=2) <= r,
        axis=0
    ) / (N - m + 1)
    return np.sum(C) / (N - m + 1)

def approximate_entropy(x, m=2, r_ratio=0.2):
    x = np.asarray(x, float)
    r = r_ratio * np.std(x)
    return np.log((_phi(x, m, r) + 1e-12) / (_phi(x, m+1, r) + 1e-12))

def sample_entropy(x, m=2, r_ratio=0.2):
    x = np.asarray(x, float)
    N = len(x)
    r = r_ratio * np.std(x)

    def _count(m):
        count = 0
        X = np.array([x[i:i+m] for i in range(N - m + 1)])
        for i in range(len(X)):
            dist = np.max(np.abs(X[i+1:] - X[i]), axis=1)
            count += np.sum(dist <= r)
        return count

    A = _count(m+1)
    B = _count(m)
    if B == 0 or A == 0:
        return np.nan
    return -np.log(A / B)

def higuchi_fd(x, kmax=6):
    x = np.asarray(x, float)
    N = len(x)
    L = []
    k_vals = range(1, kmax+1)
    for k in k_vals:
        Lk = []
        for m in range(k):
            idx = np.arange(m, N, k)
            if len(idx) < 2:
                continue
            Lm = np.sum(np.abs(np.diff(x[idx]))) * (N-1) / (k * (len(idx)-1))
            Lk.append(Lm)
        L.append(np.mean(Lk))
    L = np.log(np.array(L) + 1e-12)
    k = np.log(np.array(list(k_vals)))
    slope, _ = np.polyfit(k, L, 1)
    return -slope

def poincare_features(rr_series):
    rr = np.asarray(rr_series, float)
    rr = rr[~np.isnan(rr)]
    if len(rr) < 3:
        return np.nan, np.nan, np.nan
    x1 = rr[:-1]
    x2 = rr[1:]
    diff = x2 - x1
    sd1 = np.sqrt(np.var(diff) / 2)
    sd2 = np.sqrt(2*np.var(rr) - 0.5*np.var(diff))
    ratio = sd1 / (sd2 + 1e-12)
    return sd1, sd2, ratio

def nonlinear_features(beat, rr_for_person):
    x = beat.astype(float)
    apen = approximate_entropy(x)
    sampen = sample_entropy(x)
    hfd = higuchi_fd(x)
    sd1, sd2, ratio = poincare_features(rr_for_person)
    return [apen, sampen, hfd, sd1, sd2, ratio]


In [ ]:

def extract_ecg_features_from_person_folder(person_folder, records=RECORDS_PER_PERSON):
    all_beats = []
    all_rr = []
    for rec_name in records:
        base_path = os.path.join(person_folder, rec_name)
        if not os.path.exists(base_path + ".hea"):
            print(f"⚠️ Missing {base_path}.hea, skipping record.")
            continue
        beats, r_peaks, rr_intervals = extract_beats_from_record(base_path)
        all_beats.extend(beats)
        all_rr.extend(rr_intervals.tolist())

    if len(all_beats) == 0:
        return np.empty((0, 51))  # 51 = 15 time + 10 freq + 20 wave + 6 nonlin

    rr_for_person = np.array(all_rr)
    feature_rows = []
    for beat in all_beats:
        f_time = time_domain_features(beat)
        f_freq = frequency_domain_features(beat)
        f_wave = wavelet_features(beat)
        f_nlin = nonlinear_features(beat, rr_for_person)
        feature_rows.append(f_time + f_freq + f_wave + f_nlin)

    return np.array(feature_rows)

In [ ]:

def match_eeg_rows(eeg_matrix, n_beats):
    E = np.asarray(eeg_matrix)
    if E.shape[0] == n_beats:
        return E
    elif E.shape[0] > n_beats:
        return E[:n_beats]
    else:
        reps = int(np.ceil(n_beats / E.shape[0]))
        E_rep = np.vstack([E] * reps)
        return E_rep[:n_beats]


In [ ]:


def load_eeg_features_for_training():
    """
    Loads EEG_person_<pid>_realistic.csv from each PERSON_FOLDERS entry.
    Returns dict: { person_id(float): EEG_matrix }
    """
    eeg_by_person = {}
    for pid in PERSON_IDS_TRAIN:
        folder = PERSON_FOLDERS[pid]
        path = os.path.join(folder, f"EEG_person_{pid}_realistic.csv")
        if not os.path.exists(path):
            print(f"⚠️ No EEG CSV for Person {pid} at {path}, skipping EEG.")
            continue
        df = pd.read_csv(path)
        if "person_label" in df.columns:
            X_eeg = df.drop(columns=["person_label"]).values
        else:
            X_eeg = df.values
        eeg_by_person[float(pid)] = X_eeg
        print(f"✅ Loaded EEG for Person {pid}: shape = {X_eeg.shape}")
    return eeg_by_person

eeg_by_person = load_eeg_features_for_training()

# Build fused ECG+EEG dataset for training
X_all = []
y_all = []

for pid in PERSON_IDS_TRAIN:
    folder = PERSON_FOLDERS[pid]
    print(f"\n=== Processing Person {pid} from folder: {folder} ===")

    # ECG
    X_ecg = extract_ecg_features_from_person_folder(folder, records=RECORDS_PER_PERSON)
    if X_ecg.shape[0] == 0:
        print("❌ No ECG beats extracted, skipping this person.")
        continue
    n_beats = X_ecg.shape[0]
    print(f"  ECG beats: {n_beats}")

    # EEG
    if float(pid) not in eeg_by_person:
        print("❌ No EEG data loaded, skipping this person.")
        continue
    X_eeg_raw = eeg_by_person[float(pid)]
    X_eeg = match_eeg_rows(X_eeg_raw, n_beats)
    print(f"  EEG rows after match: {X_eeg.shape[0]}")

    X_fused = np.hstack([X_ecg, X_eeg])
    X_all.append(X_fused)
    y_all.append(np.full(n_beats, pid))

X_total = np.vstack(X_all)
y_total = np.hstack(y_all)

print("\n=== FINAL TRAIN DATA ===")
print("X_total shape:", X_total.shape)
print("y_total shape:", y_total.shape)
print("Class distribution:", np.unique(y_total, return_counts=True))


✅ Loaded EEG for Person 1: shape = (40, 38)
✅ Loaded EEG for Person 2: shape = (40, 38)
✅ Loaded EEG for Person 3: shape = (40, 38)
✅ Loaded EEG for Person 4: shape = (40, 38)
✅ Loaded EEG for Person 5: shape = (40, 38)

=== Processing Person 1 from folder: /content/drive/MyDrive/ECG EEG DATASET/Person_01 ===
  ECG beats: 40
  EEG rows after match: 40

=== Processing Person 2 from folder: /content/drive/MyDrive/ECG EEG DATASET/Person_02 ===
  ECG beats: 40
  EEG rows after match: 40

=== Processing Person 3 from folder: /content/drive/MyDrive/ECG EEG DATASET/Person_03 ===
  ECG beats: 40
  EEG rows after match: 40

=== Processing Person 4 from folder: /content/drive/MyDrive/ECG EEG DATASET/Person_04 ===
  ECG beats: 40
  EEG rows after match: 40

=== Processing Person 5 from folder: /content/drive/MyDrive/ECG EEG DATASET/Person_05 ===
  ECG beats: 40
  EEG rows after match: 40

=== FINAL TRAIN DATA ===
X_total shape: (200, 89)
y_total shape: (200,)
Class distribution: (array([1, 2, 3, 

In [ ]:
#train random forest on fused dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_total, y_total,
    test_size=0.3,
    random_state=42,
    stratify=y_total
)

clf_fusion = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)
clf_fusion.fit(X_train, y_train)
print("\nFusion model accuracy on test set:", clf_fusion.score(X_test, y_test))



Fusion model accuracy on test set: 1.0


In [ ]:
# ============================================================
# 9. CRYPTOGRAPHY: BUILD BIOMETRIC KEYS + ENCRYPT SECRET PER PERSON
# ============================================================

def build_person_templates(X_total, y_total):
    templates = {}
    for pid in sorted(np.unique(y_total)):
        X_p = X_total[y_total == pid]
        templates[int(pid)] = X_p.mean(axis=0)
    return templates

person_templates = build_person_templates(X_total, y_total)

def template_to_key(template, salt: bytes = b"ECG_EEG_BIOMETRIC_SALT"):
    t = np.asarray(template, float)
    t_min, t_max = t.min(), t.max()
    t_norm = (t - t_min) / (t_max - t_min + 1e-12)
    t_int = (t_norm * 255).astype(np.uint8)
    data_bytes = t_int.tobytes() + salt
    key = hashlib.sha256(data_bytes).digest()  # 32 bytes
    return key

person_keys = {
    pid: template_to_key(template)
    for pid, template in person_templates.items()
}

def encrypt_message(key: bytes, plaintext: str):
    iv = get_random_bytes(16)
    cipher = AES.new(key, AES.MODE_CBC, iv)
    ciphertext = cipher.encrypt(pad(plaintext.encode("utf-8"), AES.block_size))
    return iv, ciphertext

def decrypt_message(iv: bytes, ciphertext: bytes, key: bytes) -> str:
    cipher = AES.new(key, AES.MODE_CBC, iv)
    plaintext = unpad(cipher.decrypt(ciphertext), AES.block_size)
    return plaintext.decode("utf-8")

# Create a secret message per person (for demo)
iv_by_person = {}
ciphertext_by_person = {}

for pid in person_keys:
    secret = f"This is a secret message protected by ECG+EEG for person {pid}."
    iv, ct = encrypt_message(person_keys[pid], secret)
    iv_by_person[pid] = iv
    ciphertext_by_person[pid] = ct

print("\nGenerated encrypted secrets for persons:", list(person_keys.keys()))



Generated encrypted secrets for persons: [1, 2, 3, 4, 5]


In [ ]:
# ============================================================
# 10. RUNTIME: AUTH FROM FOLDER (ECG WFDB + EEG CSV)
# ============================================================

def find_eeg_csv_in_folder(person_folder):
    files = os.listdir(person_folder)
    eeg_candidates = [
        f for f in files
        if f.lower().endswith(".csv") and "eeg" in f.lower()
    ]
    if len(eeg_candidates) == 0:
        eeg_candidates = [f for f in files if f.lower().endswith(".csv")]

    if len(eeg_candidates) == 0:
        raise FileNotFoundError(f"No EEG CSV found in folder: {person_folder}")

    eeg_file = eeg_candidates[0]
    eeg_path = os.path.join(person_folder, eeg_file)
    print(f"  Using EEG file: {eeg_path}")
    return eeg_path

def load_eeg_session(eeg_csv_path):
    df = pd.read_csv(eeg_csv_path)
    if "person_label" in df.columns:
        return df.drop(columns=["person_label"]).values
    return df.values

def create_fused_session_from_folder(person_folder, records=RECORDS_PER_PERSON):
    # ECG
    X_ecg = extract_ecg_features_from_person_folder(person_folder, records=records)
    n_beats = X_ecg.shape[0]
    if n_beats == 0:
        raise ValueError(f"No beats extracted from ECG in folder: {person_folder}")
    print(f"  Extracted {n_beats} ECG beats.")

    # EEG
    eeg_csv_path = find_eeg_csv_in_folder(person_folder)
    X_eeg_raw = load_eeg_session(eeg_csv_path)
    X_eeg = match_eeg_rows(X_eeg_raw, n_beats)
    print(f"  EEG rows after match: {X_eeg.shape[0]}")

    X_fused = np.hstack([X_ecg, X_eeg])
    print("  Fused session shape:", X_fused.shape)
    return X_fused

def authenticate_and_decrypt_from_folder(
    person_folder,
    claimed_id,
    clf_fusion,
    person_keys,
    iv_by_person,
    ciphertext_by_person,
    prob_threshold=0.7,
    majority_threshold=0.6
):
    claimed_id = int(claimed_id)
    print(f"\n=== AUTHENTICATION ATTEMPT ===")
    print("Claimed ID:", claimed_id)
    print("Folder:", person_folder)

    # 1) Build fused features from folder
    X_session = create_fused_session_from_folder(person_folder)

    # 2) Classification
    preds = clf_fusion.predict(X_session)
    probs = clf_fusion.predict_proba(X_session)

    unique, counts = np.unique(preds, return_counts=True)
    majority_id = int(unique[np.argmax(counts)])
    majority_fraction = float(counts.max() / len(preds))

    max_probs = np.max(probs, axis=1)
    mean_conf = float(np.mean(max_probs))

    print("\nModel majority prediction:", majority_id)
    print("Majority fraction:", f"{majority_fraction:.3f}")
    print("Mean confidence:", f"{mean_conf:.3f}")

    accepted = (
        (majority_id == claimed_id) and
        (majority_fraction >= majority_threshold) and
        (mean_conf >= prob_threshold)
    )

    if not accepted:
        print("\n=== AUTHENTICATION RESULT: FAILED ===")
        return {
            "accepted": False,
            "claimed_id": claimed_id,
            "predicted_id": majority_id,
            "majority_fraction": majority_fraction,
            "mean_confidence": mean_conf,
            "decryption_success": False,
            "decrypted_message": None,
            "reason": "Biometric classification did not strongly support claimed ID."
        }

    print("\n=== AUTHENTICATION RESULT: SUCCESS ===")

    # 3) Decrypt secret for this person
    if claimed_id not in person_keys or claimed_id not in iv_by_person:
        print("No cryptographic data for this ID.")
        return {
            "accepted": True,
            "claimed_id": claimed_id,
            "predicted_id": majority_id,
            "majority_fraction": majority_fraction,
            "mean_confidence": mean_conf,
            "decryption_success": False,
            "decrypted_message": None,
            "reason": "No key/ciphertext stored for this ID."
        }

    key = person_keys[claimed_id]
    iv = iv_by_person[claimed_id]
    ct = ciphertext_by_person[claimed_id]

    try:
        decrypted = decrypt_message(iv, ct, key)
        print("Decrypted secret message:")
        print(decrypted)
        return {
            "accepted": True,
            "claimed_id": claimed_id,
            "predicted_id": majority_id,
            "majority_fraction": majority_fraction,
            "mean_confidence": mean_conf,
            "decryption_success": True,
            "decrypted_message": decrypted
        }
    except Exception as e:
        print("Decryption failed:", str(e))
        return {
            "accepted": True,
            "claimed_id": claimed_id,
            "predicted_id": majority_id,
            "majority_fraction": majority_fraction,
            "mean_confidence": mean_conf,
            "decryption_success": False,
            "decrypted_message": None,
            "reason": f"Decryption failed: {str(e)}"
        }


In [ ]:
#interactive authentication

claimed_id_input = int(input("Enter claimed person ID (e.g., 1–5): "))
person_folder_input = input("Enter full folder path of that person's ECG+EEG data: ")

result = authenticate_and_decrypt_from_folder(
    person_folder=person_folder_input,
    claimed_id=claimed_id_input,
    clf_fusion=clf_fusion,
    person_keys=person_keys,
    iv_by_person=iv_by_person,
    ciphertext_by_person=ciphertext_by_person,
    prob_threshold=0.7,
    majority_threshold=0.6
)

print("\nRESULT DICT:", result)


Enter claimed person ID (e.g., 1–5): 2
Enter full folder path of that person's ECG+EEG data: /content/drive/MyDrive/ECG EEG DATASET/Person_02

=== AUTHENTICATION ATTEMPT ===
Claimed ID: 2
Folder: /content/drive/MyDrive/ECG EEG DATASET/Person_02
  Extracted 40 ECG beats.
  Using EEG file: /content/drive/MyDrive/ECG EEG DATASET/Person_02/EEG_person_2_realistic.csv
  EEG rows after match: 40
  Fused session shape: (40, 89)

Model majority prediction: 2
Majority fraction: 1.000
Mean confidence: 0.940

=== AUTHENTICATION RESULT: SUCCESS ===
Decrypted secret message:
This is a secret message protected by ECG+EEG for person 2.

RESULT DICT: {'accepted': True, 'claimed_id': 2, 'predicted_id': 2, 'majority_fraction': 1.0, 'mean_confidence': 0.9395, 'decryption_success': True, 'decrypted_message': 'This is a secret message protected by ECG+EEG for person 2.'}


In [ ]:
!pip install gradio


In [ ]:
import gradio as gr
import tempfile
import zipfile
import os
import numpy as np

def get_ecg_eeg_folder_from_zip(zip_file_obj):

    if zip_file_obj is None:
        raise ValueError("No zip file uploaded.")

    zip_path = zip_file_obj.name
    extract_dir = tempfile.mkdtemp(prefix="ecg_eeg_session_")

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)

    # Try to find a folder that contains rec_1.hea
    candidate_folder = extract_dir
    found = False
    for root, dirs, files in os.walk(extract_dir):
        if any(f.startswith("rec_1") and f.endswith(".hea") for f in files):
            candidate_folder = root
            found = True
            break

    if not found:
        # Fall back: use extract_dir itself
        print("⚠️ rec_1.hea not found explicitly, using extract root:", extract_dir)

    return candidate_folder


In [ ]:
def gradio_authenticate(claimed_id, zip_file):

    try:
        claimed_id_int = int(claimed_id)
    except:
        return "❌ Invalid claimed ID. Please enter an integer (e.g., 1–5)."

    if zip_file is None:
        return "❌ Please upload a ZIP folder containing ECG WFDB + EEG CSV."

    try:
        # 1) Unzip and get the folder path
        session_folder = get_ecg_eeg_folder_from_zip(zip_file)
        print("Session folder:", session_folder)

        # 2) Call your existing auth+decrypt function
        result = authenticate_and_decrypt_from_folder(
            person_folder=session_folder,
            claimed_id=claimed_id_int,
            clf_fusion=clf_fusion,
            person_keys=person_keys,
            iv_by_person=iv_by_person,
            ciphertext_by_person=ciphertext_by_person,
            prob_threshold=0.7,
            majority_threshold=0.6
        )

        # 3) Format output nicely
        lines = []
        lines.append("=== AUTHENTICATION RESULT ===")
        lines.append(f"Claimed ID         : {result['claimed_id']}")
        lines.append(f"Predicted ID       : {result['predicted_id']}")
        lines.append(f"Accepted           : {result['accepted']}")
        lines.append(f"Majority fraction  : {result['majority_fraction']:.3f}")
        lines.append(f"Mean confidence    : {result['mean_confidence']:.3f}")

        if not result["accepted"]:
            lines.append("")
            lines.append("❌ Authentication FAILED.")
            if "reason" in result and result["reason"] is not None:
                lines.append("Reason: " + result["reason"])
            return "\n".join(lines)

        # Accepted
        lines.append("")
        if result.get("decryption_success", False):
            lines.append("✅ Authentication SUCCESS. Decrypted secret message:")
            lines.append("")
            lines.append(result.get("decrypted_message", ""))
        else:
            lines.append("✅ Authentication SUCCESS, but decryption failed or no key stored.")
            if "reason" in result and result["reason"] is not None:
                lines.append("Reason: " + result["reason"])

        return "\n".join(lines)

    except Exception as e:
        return f"❌ Error during authentication: {str(e)}"


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 🔐 ECG + EEG Biometric Authentication Demo")
    gr.Markdown(
        "1. Enter your *claimed person ID* (e.g., 1–5).\n"
        "2. Upload a **ZIP** file containing that person's ECG WFDB files (rec_1.*, rec_2.*) "
        "and EEG CSV (e.g., `EEG_person_X_realistic.csv`).\n"
        "3. Click **Authenticate** to see if the system accepts or rejects."
    )

    with gr.Row():
        claimed_id_input = gr.Number(label="Claimed Person ID", value=1, precision=0)
        zip_input = gr.File(label="Upload ZIP of ECG+EEG folder", file_types=[".zip"])

    auth_button = gr.Button("Authenticate")

    output_box = gr.Textbox(
        label="Result",
        lines=15
    )

    auth_button.click(
        fn=gradio_authenticate,
        inputs=[claimed_id_input, zip_input],
        outputs=output_box
    )

demo.launch(share=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
